# LlamaIndex — Production-Grade Data Framework for LLM Applications

---

## What Is This Notebook About?

**LlamaIndex** (formerly GPT Index) is a data framework specifically designed for connecting LLMs to your data. While LangChain is great for orchestrating multi-step LLM workflows, LlamaIndex specializes in the **data ingestion, indexing, and retrieval** part — making it the best choice for RAG applications that need to handle complex, large, and heterogeneous data sources.

Think of the difference this way:
- **LangChain** = a Swiss Army knife (does many things: chains, agents, tools, memory, RAG)
- **LlamaIndex** = a specialist surgeon's kit (does one thing supremely: indexing and querying data for LLMs)

By the end of this notebook you will understand:
- LlamaIndex's core abstractions: Documents, Nodes, Indexes, Query Engines
- Different index types and when to use each
- Advanced retrieval strategies (hybrid search, recursive retrieval)
- Document loaders for PDFs, websites, databases
- Response synthesis modes
- A mini-project: Multi-document research assistant

---

## Real-World Analogy: A Research Librarian

Imagine a university library with 10,000 books. You want an AI research assistant.

**Without LlamaIndex**: "Here's 10,000 books — figure it out" (can't fit in context window)

**With LlamaIndex**: The library has a sophisticated catalog system:
- Books are split into chapters (chunking)
- Each chapter is indexed by topic (vector embeddings)
- Cross-references between chapters (knowledge graphs)
- Summary cards for each book (tree index)
- A smart librarian (query engine) that finds exactly what you need

LlamaIndex builds and manages this entire catalog system for your AI.

---

## Prerequisites
- OpenAI SDK notebook (API calls, embeddings)
- LangChain notebook (RAG concepts, chunking)

---

## Table of Contents
1. Installation & Setup
2. Core Abstractions
3. Document Loading
4. VectorStoreIndex — The Bread and Butter
5. Index Types Comparison
6. Query Engines & Response Modes
7. Advanced Retrieval Strategies
8. LlamaIndex vs LangChain
9. Common Pitfalls
10. Mini Project: Multi-Document Research Assistant
11. Interview Q&A
12. Resources

---

## Official Resources
- **Docs**: https://docs.llamaindex.ai/
- **GitHub**: https://github.com/run-llama/llama_index
- **LlamaHub (data connectors)**: https://llamahub.ai/
- **YouTube Tutorials**: https://www.youtube.com/@llama_index
- **Discord Community**: https://discord.gg/dGcwcsnxhU

## 1. Installation & Setup

In [ ]:
# Install:
# pip install llama-index llama-index-llms-openai llama-index-embeddings-openai
# pip install llama-index-vector-stores-faiss

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

try:
    import llama_index
    from llama_index.core import (
        VectorStoreIndex, SimpleDirectoryReader, Document,
        Settings, StorageContext, load_index_from_storage
    )
    from llama_index.core.node_parser import SentenceSplitter
    from llama_index.core.schema import TextNode
    LI_AVAILABLE = True
    print(f"LlamaIndex version: {llama_index.__version__}")
except ImportError:
    LI_AVAILABLE = False
    print("LlamaIndex not installed. Run: pip install llama-index")
    print("All cells simulate output for learning purposes.")

try:
    from llama_index.llms.openai import OpenAI as LlamaOpenAI
    from llama_index.embeddings.openai import OpenAIEmbedding
    OPENAI_LI = True
except ImportError:
    OPENAI_LI = False

API_KEY = os.getenv('OPENAI_API_KEY', '')
HAS_KEY = bool(API_KEY)
CAN_CALL = LI_AVAILABLE and OPENAI_LI and HAS_KEY

print(f"\nLlamaIndex: {'✓' if LI_AVAILABLE else '✗'}  OpenAI integration: {'✓' if OPENAI_LI else '✗'}  API Key: {'✓' if HAS_KEY else '✗'}")

if CAN_CALL:
    # Configure global LLM and embedding model
    Settings.llm = LlamaOpenAI(model='gpt-3.5-turbo', temperature=0.1)
    Settings.embed_model = OpenAIEmbedding(model='text-embedding-3-small')
    Settings.chunk_size = 512
    Settings.chunk_overlap = 50
    print("LlamaIndex configured with OpenAI!")
else:
    print("Simulating all LlamaIndex outputs.")

## 2. Core Abstractions

LlamaIndex has a clean hierarchy of abstractions:

```
Documents (raw files)
    ↓ [Parsing / Splitting]
Nodes (text chunks with metadata)
    ↓ [Embedding]
Index (efficient data structure for retrieval)
    ↓ [Querying]
Retriever (finds relevant Nodes)
    ↓ [Synthesis]
Response Synthesizer (LLM generates answer from retrieved Nodes)
    ↓
Query Engine (combines Retriever + Synthesizer)
```

| Abstraction | Description | Analogy |
|-------------|-------------|--------|
| **Document** | Raw source file (PDF, URL, string) | A book |
| **Node** | A chunk of a Document with metadata | A page or paragraph |
| **Index** | Organized data structure over Nodes | A card catalog |
| **Retriever** | Fetches relevant Nodes for a query | A librarian |
| **ResponseSynthesizer** | LLM creates answer from Nodes | The researcher |
| **QueryEngine** | Retriever + Synthesizer combined | The library system |

In [ ]:
# ── Core Abstractions Demo ────────────────────────────────────────────

# Create Documents manually
raw_texts = [
    """Artificial Intelligence (AI) is the simulation of human intelligence processes by machines.
    AI applications include expert systems, natural language processing, speech recognition,
    and machine vision. The key branches are machine learning, deep learning, and NLP.""",

    """Machine Learning (ML) is a subset of AI that allows computers to learn from data without
    explicit programming. Types include supervised learning (labeled data), unsupervised learning
    (finding patterns), and reinforcement learning (learning through rewards).""",

    """Deep Learning uses artificial neural networks with many layers (hence 'deep'). It excels
    at image recognition, speech processing, and natural language understanding. The key
    architectures are CNNs (images), RNNs/Transformers (sequences), and GANs (generation).""",
]

if LI_AVAILABLE:
    # Create Document objects (with metadata)
    documents = [
        Document(
            text=text,
            metadata={
                'topic': ['AI', 'ML', 'Deep Learning'][i],
                'doc_id': f'doc_{i}',
                'source': 'ai_textbook.pdf'
            }
        )
        for i, text in enumerate(raw_texts)
    ]

    print("=== LlamaIndex Core Objects ===")
    print(f"\nDocument 0:")
    print(f"  Text: '{documents[0].text[:80]}...'")
    print(f"  Metadata: {documents[0].metadata}")
    print(f"  Doc ID: {documents[0].doc_id}")

    # Parse into Nodes (chunks)
    parser = SentenceSplitter(chunk_size=150, chunk_overlap=20)
    nodes = parser.get_nodes_from_documents(documents)

    print(f"\n3 Documents → {len(nodes)} Nodes (chunks)")
    print(f"\nNode 0:")
    print(f"  Text: '{nodes[0].text[:100]}...'")
    print(f"  Metadata: {nodes[0].metadata}")
    print(f"  Node ID: {nodes[0].node_id}")
    if nodes[0].next_node:
        print(f"  Next node: {nodes[0].next_node.node_id}  ← nodes are linked!")  

else:
    print("=== LlamaIndex Core Objects (simulated) ===")
    print()
    print("Documents: Raw source files with text + metadata")
    print("  doc = Document(text='...', metadata={'source': 'textbook.pdf', 'page': 42})")
    print()
    print("Nodes: Chunks of documents")
    print("  parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)")
    print("  nodes = parser.get_nodes_from_documents(documents)")
    print("  → Each node has: text, metadata (inherited), node_id, relationships (prev/next)")
    print()
    print("Key feature: Nodes preserve relationships!")
    print("  node.prev_node → link to previous chunk")
    print("  node.next_node → link to next chunk")
    print("  Useful for 'retrieve this chunk + its neighbors' strategies")

# Visualize the abstraction hierarchy
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

levels = [
    ('Documents\n(raw files)', 0.85, '#3498db'),
    ('Nodes\n(text chunks + metadata)', 0.65, '#9b59b6'),
    ('Index\n(organized for retrieval)', 0.45, '#e74c3c'),
    ('Query Engine\n(ask questions)', 0.25, '#2ecc71'),
]

for label, y, color in levels:
    rect = mpatches.FancyBboxPatch((0.2, y-0.07), 0.6, 0.13,
                                    boxstyle='round,pad=0.02',
                                    facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(0.5, y, label, ha='center', va='center', fontsize=11, fontweight='bold', color='white')

    if y > 0.25:
        ax.annotate('', xy=(0.5, y - 0.09), xytext=(0.5, y - 0.07 + 0.13 - 0.07),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(0.5, 0.96, 'LlamaIndex Abstraction Hierarchy', ha='center',
        fontsize=14, fontweight='bold')
ax.text(0.85, 0.88, 'PDF, TXT, URL,\nSQL, Notion...', ha='center', fontsize=9, color='#3498db')
ax.text(0.85, 0.68, '512-char chunks\nwith links', ha='center', fontsize=9, color='#9b59b6')
ax.text(0.85, 0.48, 'Vector, Tree,\nList, KG...', ha='center', fontsize=9, color='#e74c3c')
ax.text(0.85, 0.28, 'Retrieve + Synthesize\n= Answer', ha='center', fontsize=9, color='#2ecc71')

plt.tight_layout()
plt.savefig('/tmp/llamaindex_hierarchy.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. VectorStoreIndex — The Most Common Index

The `VectorStoreIndex` is the most widely used index. It:
1. Embeds all nodes into vectors
2. Stores them in a vector database (FAISS, Chroma, Pinecone, etc.)
3. On query: embed the query, find the k most similar nodes (top-k search)
4. Pass retrieved nodes to the LLM for synthesis

**The 3-line RAG:**
```python
index = VectorStoreIndex.from_documents(documents)  # Build
query_engine = index.as_query_engine()              # Create query engine
response = query_engine.query("Your question here") # Ask
```

That's it! LlamaIndex handles chunking, embedding, storing, and retrieval.

In [ ]:
# ── VectorStoreIndex: Build and Query ─────────────────────────────────

if CAN_CALL and LI_AVAILABLE:
    # Build index
    print("Building VectorStoreIndex...")
    index = VectorStoreIndex.from_documents(
        documents,
        show_progress=True
    )
    print("Index built!")

    # Create query engine
    query_engine = index.as_query_engine(
        similarity_top_k=3,      # Retrieve top 3 most similar nodes
        response_mode='compact', # Compact response synthesis
    )

    # Query
    questions = [
        "What is the difference between AI and machine learning?",
        "What are the types of machine learning?",
        "What neural network architectures are used in deep learning?",
    ]

    print("\n=== Querying the Index ===")
    for q in questions:
        response = query_engine.query(q)
        print(f"\nQ: {q}")
        print(f"A: {str(response)[:200]}")
        if hasattr(response, 'source_nodes'):
            print(f"Sources: {[n.node.metadata.get('topic', 'unknown') for n in response.source_nodes]}")

else:
    print("=== VectorStoreIndex (simulated) ===")
    print()
    print("# 3-line RAG:")
    print("index = VectorStoreIndex.from_documents(documents)")
    print("query_engine = index.as_query_engine(similarity_top_k=3)")
    print("response = query_engine.query('What is machine learning?')")
    print()
    print("Q: What is the difference between AI and machine learning?")
    print("A: AI is the broad concept of machines simulating human intelligence, while machine")
    print("   learning is a specific subset that enables computers to learn from data without")
    print("   explicit programming. All machine learning is AI, but not all AI is machine learning.")
    print("Sources: ['AI', 'ML']")
    print()
    print("Q: What are the types of machine learning?")
    print("A: There are three main types: supervised learning (trained on labeled data),")
    print("   unsupervised learning (finds patterns in unlabeled data), and reinforcement")
    print("   learning (learns through rewards and penalties).")
    print("Sources: ['ML']")
    print()

    # Show what the response object contains
    print("Response object structure:")
    print("  response.response          → The generated text answer")
    print("  response.source_nodes      → List of retrieved nodes used")
    print("  response.source_nodes[0].node.text     → Source text")
    print("  response.source_nodes[0].score         → Similarity score")
    print("  response.source_nodes[0].node.metadata → Source metadata")

## 4. Index Types — Choosing the Right Data Structure

LlamaIndex offers several index types, each with different strengths:

| Index | How It Works | Best For | Trade-off |
|-------|-------------|----------|----------|
| **VectorStoreIndex** | Embed + k-NN search | Most use cases | Fast but misses non-semantic matches |
| **SummaryIndex** | Summarize all nodes | Summarization tasks | High cost (reads all nodes) |
| **TreeIndex** | Hierarchical tree of summaries | Long documents, hierarchical Q&A | Complex build |
| **KeywordTableIndex** | Keyword extraction + inverted index | Keyword-specific queries | Misses synonyms |
| **KnowledgeGraphIndex** | Extract entity-relationship triples | Complex multi-hop reasoning | Expensive build |

### Tree Index — For Very Long Documents
```
Level 0: All raw nodes (leaf nodes)
Level 1: Summary of groups of nodes
Level 2: Summary of summaries
Level 3: Top-level summary

Query: Start at root, navigate DOWN to find relevant leaves
```
Useful when a document is longer than the context window — you can't embed the whole thing and the vector search might miss global context.

In [ ]:
# ── Index Types Comparison Visualization ─────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('LlamaIndex: Index Types Comparison', fontsize=14, fontweight='bold')

# 1. VectorStoreIndex
ax = axes[0]
ax.axis('off')
ax.set_title('VectorStoreIndex\n(Most Common)', fontweight='bold', color='#3498db')

# Draw nodes in 2D embedding space
np.random.seed(42)
n_nodes = 12
node_positions = np.random.randn(n_nodes, 2) * 0.3
# Cluster them
node_positions[:4] += [0.5, 0.5]
node_positions[4:8] += [-0.5, 0.3]
node_positions[8:] += [0.1, -0.6]

colors = ['#3498db'] * 4 + ['#e74c3c'] * 4 + ['#2ecc71'] * 4
ax.scatter(node_positions[:, 0], node_positions[:, 1], c=colors, s=100, zorder=5)

# Query vector
query_pos = np.array([0.4, 0.4])
ax.scatter(*query_pos, marker='*', c='gold', s=300, zorder=10, label='Query')
circle = plt.Circle(query_pos, 0.25, fill=False, color='gold', linestyle='--', linewidth=2)
ax.add_patch(circle)
ax.text(*query_pos, '  Query', fontsize=9, color='goldenrod')

ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_xlabel('Semantic Dimension 1')
ax.set_ylabel('Semantic Dimension 2')
ax.text(0, -1.4, 'k-NN: Find nodes closest\nto query in vector space',
        ha='center', fontsize=8, style='italic')
ax.grid(True, alpha=0.2)

# 2. TreeIndex
ax = axes[1]
ax.axis('off')
ax.set_title('TreeIndex\n(Long Documents)', fontweight='bold', color='#e74c3c')

# Draw tree
tree_nodes = [
    (0.5, 0.9, 'Root\nSummary', '#e74c3c', 0.15),  # root
    (0.25, 0.65, 'Summary\nA', '#e67e22', 0.12),
    (0.75, 0.65, 'Summary\nB', '#e67e22', 0.12),
    (0.12, 0.35, 'Node\n1', '#f39c12', 0.10),
    (0.37, 0.35, 'Node\n2', '#f39c12', 0.10),
    (0.62, 0.35, 'Node\n3', '#f39c12', 0.10),
    (0.87, 0.35, 'Node\n4', '#f39c12', 0.10),
]

edges = [(0,1), (0,2), (1,3), (1,4), (2,5), (2,6)]
for i, j in edges:
    ax.plot([tree_nodes[i][0], tree_nodes[j][0]],
            [tree_nodes[i][1], tree_nodes[j][1]], 'k-', linewidth=1.5, alpha=0.5)

for x, y, label, color, size in tree_nodes:
    circle = plt.Circle((x, y), size, color=color, alpha=0.8)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=7, fontweight='bold', color='white')

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0.1, 1.1)
ax.text(0.5, 0.05, 'Query navigates top-down:\nRoot → Summary → Leaf Nodes',
        ha='center', fontsize=8, style='italic')

# Add query path
ax.annotate('Query →', xy=(0.75, 0.65), xytext=(0.9, 0.8),
            arrowprops=dict(arrowstyle='->', color='gold', lw=2))

# 3. KnowledgeGraph
ax = axes[2]
ax.axis('off')
ax.set_title('KnowledgeGraphIndex\n(Relationships)', fontweight='bold', color='#9b59b6')

kg_nodes = {
    'AI': (0.5, 0.8),
    'ML': (0.2, 0.5),
    'DL': (0.5, 0.3),
    'NLP': (0.8, 0.5),
    'GPT': (0.8, 0.2),
    'CNN': (0.2, 0.2),
}

kg_edges = [
    ('AI', 'ML', 'includes'),
    ('AI', 'NLP', 'includes'),
    ('ML', 'DL', 'includes'),
    ('DL', 'CNN', 'type'),
    ('NLP', 'GPT', 'uses'),
    ('DL', 'GPT', 'powers'),
]

for src, dst, label in kg_edges:
    x1, y1 = kg_nodes[src]
    x2, y2 = kg_nodes[dst]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#9b59b6', lw=1.5))
    mx, my = (x1+x2)/2, (y1+y2)/2
    ax.text(mx, my, label, fontsize=7, color='#555', ha='center')

for name, (x, y) in kg_nodes.items():
    rect = mpatches.FancyBboxPatch((x-0.08, y-0.05), 0.16, 0.1,
                                    boxstyle='round,pad=0.02',
                                    facecolor='#9b59b6', edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, name, ha='center', va='center', fontsize=9, fontweight='bold', color='white')

ax.set_xlim(0, 1)
ax.set_ylim(0.05, 1)
ax.text(0.5, 0.05, 'Multi-hop: AI → includes → ML → includes → DL → powers → GPT',
        ha='center', fontsize=7, style='italic')

plt.tight_layout()
plt.savefig('/tmp/llamaindex_indexes.png', dpi=100, bbox_inches='tight')
plt.show()

print("Index Selection Guide:")
print("  Most RAG apps: VectorStoreIndex (fast, works well for most tasks)")
print("  Summarization: SummaryIndex (reads all nodes to generate summary)")
print("  Very long books: TreeIndex (hierarchical navigation)")
print("  Multi-hop Q&A: KnowledgeGraphIndex ('Who is the CEO of the company that makes GPT?')")

## 5. Query Engines & Response Modes

In [ ]:
# ── Response Modes Explained ──────────────────────────────────────────

print("=" * 65)
print(" LlamaIndex Response Synthesis Modes")
print("=" * 65)
print()

response_modes = {
    'compact': {
        'description': 'Compresses retrieved nodes into minimal tokens, then synthesizes.',
        'how': 'Concatenate nodes → truncate to fit context → one LLM call',
        'best_for': 'Most use cases — balances speed and quality',
        'cost': '1 LLM call',
        'tradeoff': 'May miss nuances if context is truncated'
    },
    'refine': {
        'description': 'Processes nodes one by one, refining the answer as it goes.',
        'how': 'Node1 → answer1 → Node2 → refine answer1 → answer2 → ...',
        'best_for': 'Detailed answers that need all retrieved context',
        'cost': 'N LLM calls (one per node)',
        'tradeoff': 'Expensive — N API calls instead of 1'
    },
    'tree_summarize': {
        'description': 'Summarizes groups of nodes, then summarizes summaries.',
        'how': 'Group nodes → summarize each group → summarize summaries → final answer',
        'best_for': 'Very long documents, when all nodes are important',
        'cost': 'O(log N) LLM calls',
        'tradeoff': 'Slower than compact, but handles more context'
    },
    'no_text': {
        'description': 'Returns raw retrieved nodes WITHOUT LLM synthesis.',
        'how': 'Retrieve → return nodes directly (no generation)',
        'best_for': 'When you want to do your own synthesis, or just retrieve',
        'cost': '0 LLM calls (just embedding for retrieval)',
        'tradeoff': 'No natural language answer — you get raw chunks'
    },
}

for mode, info in response_modes.items():
    print(f"Mode: '{mode}'")
    print(f"  {info['description']}")
    print(f"  How: {info['how']}")
    print(f"  Best for: {info['best_for']}")
    print(f"  Cost: {info['cost']}")
    print(f"  Trade-off: {info['tradeoff']}")
    print()

print("Usage:")
print("  query_engine = index.as_query_engine(response_mode='refine')")
print("  # or")
print("  query_engine = index.as_query_engine(response_mode='tree_summarize')")

## 6. Advanced Retrieval Strategies

The simplest retrieval (top-k similarity search) often has problems:
- **Recency bias**: recent documents dominate
- **Exact match misses**: query "what does the CEO earn" doesn't match "salary of the president"
- **Context loss**: a retrieved chunk makes no sense without the surrounding text

Advanced strategies in LlamaIndex:

| Strategy | How It Works | When To Use |
|----------|-------------|-------------|
| **HyDE** | Generate a hypothetical document for the query, embed that | When query phrasing doesn't match document style |
| **Sentence Window** | Retrieve sentence, but use surrounding sentences as context | When precise retrieval + context matters |
| **Auto-merging** | Retrieve small chunks, merge into parent if many children match | Hierarchical docs |
| **Hybrid Search** | BM25 (keyword) + Vector search, combine scores | Need both exact + semantic matching |
| **MMR** | Maximum Marginal Relevance: diverse results | Avoid retrieving duplicate chunks |

In [ ]:
# ── Advanced Retrieval: HyDE (Hypothetical Document Embeddings) ────────
#
# Problem: Query 'revenue growth' doesn't semantically match
#          'our sales increased by 23% year-over-year'
#
# HyDE Solution:
#   1. Use LLM to generate a HYPOTHETICAL document that would answer the query
#   2. Embed the hypothetical doc (not the query!)
#   3. Search for similar REAL docs using the hypothetical embedding
#
# The hypothetical doc uses similar language to real docs → better retrieval

def simulate_hyde(query):
    """Simulate the HyDE process."""
    # Step 1: Generate hypothetical document
    hypothetical_docs = {
        "What is the revenue growth?": (
            "Our company experienced significant revenue growth this fiscal year. "
            "Sales increased by 23% year-over-year to reach $45.2M. "
            "Growth was driven primarily by enterprise clients and new product launches."
        ),
        "How does machine learning work?": (
            "Machine learning works by training algorithms on labeled data to identify patterns. "
            "The model adjusts its parameters through gradient descent to minimize prediction error. "
            "After training, the model can make predictions on new unseen data."
        ),
    }
    return hypothetical_docs.get(query, f"[Hypothetical document answering: {query}]")


print("=== HyDE: Hypothetical Document Embeddings ===")
print()

query = "What is the revenue growth?"
hypothetical = simulate_hyde(query)

print(f"Original query: '{query}'")
print(f"  → Embed query directly and search")
print(f"     Problem: query is short and abstract")
print()
print("HyDE approach:")
print(f"  Step 1: LLM generates hypothetical doc:")
print(f"    '{hypothetical}'")
print(f"  Step 2: Embed the HYPOTHETICAL DOC (not the query)")
print(f"  Step 3: Search for real docs similar to this hypothetical doc")
print()
print("Why it works:")
print("  The hypothetical doc is in the SAME STYLE as real documents")
print("  '23% year-over-year' → matches real finance docs better than 'revenue growth'")
print()

# LlamaIndex HyDE code
print("LlamaIndex HyDE code:")
print('''
from llama_index.core.indices.query.query_transform.base import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine

# Wrap query engine with HyDE transform
hyde = HyDEQueryTransform(include_original=True)
hyde_engine = TransformQueryEngine(query_engine, hyde)

response = hyde_engine.query("What is the revenue growth?")
# HyDE internally: query → hypothetical doc → embed → retrieve → synthesize
''')

# Visualize HyDE
fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')

steps = [
    ('User Query\n"revenue growth"', 0.08, '#e74c3c'),
    ('LLM generates\nhypothetical doc', 0.28, '#9b59b6'),
    ('Embed hypothetical\ndoc (not query!)', 0.48, '#3498db'),
    ('Retrieve similar\nreal documents', 0.68, '#2ecc71'),
    ('Synthesize\nanswer', 0.88, '#f39c12'),
]

for label, x, color in steps:
    rect = mpatches.FancyBboxPatch((x-0.09, 0.25), 0.18, 0.5,
                                    boxstyle='round,pad=0.02',
                                    facecolor=color, edgecolor='white', linewidth=2, alpha=0.85)
    ax.add_patch(rect)
    ax.text(x, 0.5, label, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if x < 0.88:
        ax.annotate('', xy=(x+0.11, 0.5), xytext=(x+0.09, 0.5),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.text(0.5, 0.9, 'HyDE: Hypothetical Document Embeddings Flow',
        ha='center', fontsize=13, fontweight='bold')
ax.text(0.5, 0.1, 'Key insight: search for docs similar to "what an answer looks like", not similar to "the question"',
        ha='center', fontsize=9, color='gray', style='italic')

plt.tight_layout()
plt.savefig('/tmp/llamaindex_hyde.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Persisting and Loading Indexes

In [ ]:
# ── Saving & Loading Indexes ──────────────────────────────────────────
#
# Building an index (embedding all documents) is expensive.
# Save it once, load it every time — no need to re-embed!

print("=== Index Persistence ===")
print()

if CAN_CALL and LI_AVAILABLE:
    import os
    PERSIST_DIR = '/tmp/llamaindex_storage'

    if not os.path.exists(PERSIST_DIR):
        print("Building index for first time...")
        index = VectorStoreIndex.from_documents(documents)
        index.storage_context.persist(persist_dir=PERSIST_DIR)
        print(f"Index saved to {PERSIST_DIR}")
    else:
        print("Loading existing index (no re-embedding needed!)")
        storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
        index = load_index_from_storage(storage_context)
        print("Index loaded!")

    query_engine = index.as_query_engine()
    r = query_engine.query("What is deep learning?")
    print(f"\nTest query: 'What is deep learning?'")
    print(f"Answer: {str(r)[:200]}")

else:
    print("# Save index (do this once after building):")
    print("index = VectorStoreIndex.from_documents(documents)")
    print("index.storage_context.persist(persist_dir='./my_index')")
    print("# Files saved:")
    print("#   ./my_index/docstore.json     → document metadata")
    print("#   ./my_index/vector_store.json → embeddings")
    print("#   ./my_index/index_store.json  → index structure")
    print()
    print("# Load index on next run (no API calls needed!):")
    print("storage_context = StorageContext.from_defaults(persist_dir='./my_index')")
    print("index = load_index_from_storage(storage_context)")
    print("query_engine = index.as_query_engine()")
    print()
    print("Cost savings:")
    print("  Building index: $0.0002 per 1000 tokens (text-embedding-3-small)")
    print("  Loading saved: $0.00 — no embedding calls!")
    print("  A 100-page PDF embedded once saves ~$0.02; across 1000 queries that's huge.")
    print()
    print("For production: use Pinecone, Weaviate, or Chroma instead of FAISS")
    print("  → They persist automatically in the cloud")
    print("  → Multiple servers can share the same index")
    print("  → Built-in metadata filtering")

## 8. Common Pitfalls

In [ ]:
print("=" * 68)
print(" LlamaIndex Common Pitfalls")
print("=" * 68)

pitfalls = [
    {
        "title": "1. Not persisting indexes → re-embedding on every restart",
        "fix": "index.storage_context.persist(persist_dir='./storage')",
        "why": "Embedding costs money and time. Save once, load many times."
    },
    {
        "title": "2. Default chunk_size=1024 is often too large",
        "fix": "Settings.chunk_size = 512  # Or tune for your use case",
        "why": "Large chunks have mixed topics → poor retrieval precision."
    },
    {
        "title": "3. Not setting Settings globally — using deprecated per-call config",
        "fix": "from llama_index.core import Settings; Settings.llm = ...; Settings.embed_model = ...",
        "why": "LlamaIndex v0.10+ uses Settings instead of ServiceContext."
    },
    {
        "title": "4. Using default embedding model (text-davinci-003 → deprecated)",
        "fix": "Settings.embed_model = OpenAIEmbedding(model='text-embedding-3-small')",
        "why": "Default model changes between versions. Explicitly set for reproducibility."
    },
    {
        "title": "5. Ignoring metadata filters → irrelevant docs retrieved",
        "fix": "Use MetadataFilter: retriever = index.as_retriever(filters=MetadataFilters(...))",
        "why": "Without filters, a query about 2024 might retrieve 2020 documents."
    },
    {
        "title": "6. Building indexes on too many documents at once → OOM",
        "fix": "Process in batches: for batch in chunks(documents, 100): index.insert_nodes(batch)",
        "why": "Embedding 10k documents at once loads all into memory."
    },
]

for p in pitfalls:
    print(f"\n{'─'*68}")
    print(f"  {p['title']}")
    print(f"  Fix:  {p['fix']}")
    print(f"  Why:  {p['why']}")

print(f"\n{'='*68}")

## 9. Mini Project: Multi-Document Research Assistant

In [ ]:
# ── Mini Project: Multi-Document Research Assistant ───────────────────
#
# A research assistant that:
# 1. Indexes multiple documents on different topics
# 2. Routes queries to the right document sub-indexes
# 3. Synthesizes answers from multiple sources
# 4. Provides source attribution

# This implements the LlamaIndex RouterQueryEngine pattern
# (without requiring the actual library in this demo)

class MultiDocResearchAssistant:
    """
    Simulates LlamaIndex's RouterQueryEngine pattern.

    Real implementation uses:
      - SubQuestionQueryEngine for decomposing complex queries
      - RouterQueryEngine to route to the right index
      - VectorStoreIndex per document collection
    """

    # Simulated document collections
    COLLECTIONS = {
        'finance': {
            'description': 'Financial reports, revenue, profit, costs',
            'keywords': ['revenue', 'profit', 'cost', 'financial', 'earnings', 'stock', 'sales'],
            'documents': [
                "Q3 2024: Revenue grew 23% YoY to $45.2M. Gross margin improved to 72%.",
                "Operating costs increased 15% due to R&D investment. Net income: $8.1M.",
                "Full year guidance: revenue $180-190M, with 25% growth expected in enterprise segment."
            ]
        },
        'hr': {
            'description': 'HR policies, benefits, vacation, compensation',
            'keywords': ['employee', 'vacation', 'benefits', 'salary', 'hiring', 'leave', 'remote'],
            'documents': [
                "Vacation policy: 20 days PTO per year, 10 sick days. Rolls over up to 5 days.",
                "Remote work: 3 days/week in office required. Full remote for senior ICs.",
                "Benefits: health insurance (employer pays 90%), 401k match up to 4%, $2000/yr learning budget."
            ]
        },
        'product': {
            'description': 'Product features, roadmap, technical specifications',
            'keywords': ['feature', 'product', 'release', 'api', 'integration', 'deployment', 'technical'],
            'documents': [
                "DataFlow Pro v3.0: adds real-time streaming, 50+ new connectors, and improved ML pipeline support.",
                "Q4 roadmap: AI-powered anomaly detection, automated schema inference, multi-cloud support.",
                "API v2: REST and GraphQL support, rate limit 1000 req/min, SLA 99.9% uptime."
            ]
        }
    }

    def route_query(self, query):
        """Route query to the most relevant collection."""
        query_lower = query.lower()
        scores = {}
        for collection, data in self.COLLECTIONS.items():
            score = sum(1 for kw in data['keywords'] if kw in query_lower)
            scores[collection] = score
        best = max(scores, key=scores.get)
        return best, scores

    def answer(self, query):
        """Answer a query by routing and retrieving."""
        collection, routing_scores = self.route_query(query)
        docs = self.COLLECTIONS[collection]['documents']

        # Simple keyword-based retrieval within collection
        query_words = set(query.lower().split())
        scored_docs = []
        for doc in docs:
            doc_words = set(doc.lower().split())
            overlap = len(query_words & doc_words)
            scored_docs.append((overlap, doc))
        scored_docs.sort(reverse=True)
        relevant_docs = [doc for _, doc in scored_docs[:2]]

        return {
            'routed_to': collection,
            'routing_scores': routing_scores,
            'retrieved': relevant_docs,
            'answer_basis': relevant_docs[0] if relevant_docs else "No relevant content found."
        }


assistant = MultiDocResearchAssistant()

test_queries = [
    "What is our revenue growth this quarter?",
    "How many vacation days do employees get?",
    "What new features are coming in Q4?",
    "What's the API rate limit?",
]

print("=" * 65)
print(" Multi-Document Research Assistant")
print("=" * 65)
print()

for query in test_queries:
    result = assistant.answer(query)
    print(f"Query: '{query}'")
    print(f"  Routed to: {result['routed_to'].upper()} collection")
    print(f"  Routing scores: {result['routing_scores']}")
    print(f"  Key source: '{result['answer_basis'][:80]}...'")
    print()

print("Production LlamaIndex version of this pattern:")
print('''
from llama_index.core.tools import QueryEngineTool
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

# Create one index per document collection
finance_index = VectorStoreIndex.from_documents(finance_docs)
hr_index = VectorStoreIndex.from_documents(hr_docs)
product_index = VectorStoreIndex.from_documents(product_docs)

# Wrap each in a tool with a description
tools = [
    QueryEngineTool.from_defaults(
        finance_index.as_query_engine(),
        description="Financial reports: revenue, profit, earnings"
    ),
    QueryEngineTool.from_defaults(
        hr_index.as_query_engine(),
        description="HR policies: vacation, benefits, compensation"
    ),
    QueryEngineTool.from_defaults(
        product_index.as_query_engine(),
        description="Product docs: features, roadmap, API specs"
    ),
]

# Router uses LLM to pick the right tool
router = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=tools
)

response = router.query("What is our revenue growth?")
# → Automatically routed to finance index!
''')

## 10. Interview Q&A

---

### Q1: What is LlamaIndex and how is it different from LangChain?
**A**: LlamaIndex is a data framework specialized for connecting LLMs to data sources. Its primary focus is sophisticated data indexing and retrieval — multiple index types, advanced chunking, metadata filtering, and query transformations. LangChain is broader — it handles chains, agents, tools, memory, and RAG, but its RAG capabilities are simpler than LlamaIndex's. Many production systems use both: LlamaIndex for data indexing and retrieval, LangChain for agent orchestration. If your main challenge is "how do I efficiently query my documents?", choose LlamaIndex. If it's "how do I build a multi-step LLM workflow?", choose LangChain.

---

### Q2: What is a Node in LlamaIndex?
**A**: A Node is LlamaIndex's fundamental unit of data — a chunk of text with metadata and relationships. Each document is split into Nodes during indexing. Nodes inherit metadata from their parent document (source file, page number, etc.) and also know their neighbors (`prev_node`, `next_node`). This relationship awareness enables advanced strategies like "sentence window retrieval" (retrieve a Node, but return it plus its neighbors for context). Nodes also have unique IDs used for deduplication and updates.

---

### Q3: When would you use TreeIndex instead of VectorStoreIndex?
**A**: TreeIndex is useful when: (1) Documents are very long (100+ pages) and you need hierarchical navigation rather than flat k-NN search, (2) You need comprehensive summarization of an entire document (TreeIndex reads all nodes), (3) Questions require understanding the document's overall structure ("what's the main theme of chapter 5?"). VectorStoreIndex is better for most RAG use cases because it's faster and cheaper. The trade-off: TreeIndex builds summaries at multiple levels during indexing (expensive), but queries navigate the hierarchy efficiently.

---

### Q4: What is HyDE and why is it powerful?
**A**: HyDE (Hypothetical Document Embeddings) addresses the mismatch between query style and document style. Problem: a user query "revenue growth" is phrased abstractly, but real documents say "revenue grew 23% year-over-year". These embeddings may be dissimilar. Solution: use an LLM to generate a hypothetical document that WOULD answer the query, then embed that hypothetical document for search. The hypothetical doc uses similar language to real docs, so it finds better matches. Downside: requires one extra LLM call per query. But often dramatically improves retrieval precision.

---

### Q5: What is a vector store and which should you use in production?
**A**: A vector store is a database optimized for storing and searching high-dimensional vectors (embeddings). It supports k-nearest-neighbor (k-NN) search efficiently. Options: **FAISS** (in-memory, local, no infrastructure needed, good for dev/small datasets), **Chroma** (local or hosted, easy to set up), **Pinecone** (managed cloud service, scales to billions of vectors), **Weaviate** (open-source, self-hosted or cloud, rich filtering), **Qdrant** (fast, open-source, good metadata filtering). For development: FAISS or Chroma. For production with scale: Pinecone or Weaviate.

---

### Q6: What are the key metrics for evaluating a RAG system?
**A**: RAG evaluation has two stages: (1) **Retrieval quality**: Are the right chunks being retrieved? Metrics: Precision@k (what fraction of retrieved chunks are relevant?), Recall (what fraction of relevant chunks were retrieved?). (2) **Generation quality**: Does the answer correctly use the retrieved context? Metrics: Faithfulness (is every claim in the answer supported by retrieved context?), Answer relevancy (does the answer actually address the question?), Context precision (does the model use the context efficiently?). Tools: RAGAS library (automated evaluation), LlamaIndex Evaluation modules, human evaluation for production.

## 11. Resources

### Official
- **LlamaIndex Docs**: https://docs.llamaindex.ai/
- **LlamaHub (data connectors)**: https://llamahub.ai/
- **GitHub**: https://github.com/run-llama/llama_index
- **Discord**: https://discord.gg/dGcwcsnxhU

### Tutorials
- **LlamaIndex YouTube**: https://www.youtube.com/@llama_index
- **Advanced RAG Techniques**: https://www.youtube.com/watch?v=TRjq7t2Ms5I
- **Jerry Liu (founder) talks**: https://www.youtube.com/results?search_query=jerry+liu+llamaindex

### Papers
- **HyDE**: https://arxiv.org/abs/2212.10496
- **RAGAS (RAG evaluation)**: https://arxiv.org/abs/2309.15217
- **Self-RAG**: https://arxiv.org/abs/2310.11511

---

## Summary

| Concept | Takeaway |
|---------|----------|
| Documents → Nodes | Split docs into chunks with metadata + relationships |
| VectorStoreIndex | Embed + k-NN search — default for most RAG |
| TreeIndex | Hierarchical for long docs/summarization |
| QueryEngine | Retriever + Synthesizer = ask questions |
| Response modes | compact (fast), refine (thorough), tree_summarize (long docs) |
| HyDE | Embed hypothetical answer, not the query |
| Persistence | Save index once, load forever — save API costs |
| vs LangChain | LlamaIndex = data specialist; LangChain = orchestration toolkit |

**Next**: vLLM — serve open-source LLMs at production scale with high throughput!